# `mosaic(grid_id=...)` - Sentinel-2 mosaics for full MGRS tiles

Use `grid_id` when you want a complete Sentinel-2 MGRS tile. Full-tile jobs are much larger than bounds or AOI jobs, so this notebook uses lower resolution and SCL cloud masking for the first export example.

In [1]:
from pathlib import Path
import json

import numpy as np
from matplotlib import pyplot as plt
import rasterio as rio

from s2mosaic import mosaic


def plot_rgb(array, title=None, figsize=(8, 6)):
    """Display B04/B03/B02-style arrays with a robust percentile stretch."""
    rgb = np.moveaxis(array[:3], 0, -1).astype(float)
    valid = rgb[np.isfinite(rgb) & (rgb > 0)]
    if valid.size == 0:
        raise ValueError("No positive finite pixels to display")
    lo, hi = np.nanpercentile(valid, [2, 98])
    if not np.isfinite(hi - lo) or hi <= lo:
        hi = np.nanmax(valid)
        lo = np.nanmin(valid)
    rgb_disp = np.clip((rgb - lo) / max(hi - lo, 1), 0, 1)

    plt.figure(figsize=figsize)
    plt.imshow(rgb_disp)
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()

## 1. Export a visual mosaic

This cell writes a full-tile visual mosaic. Expect minutes rather than seconds on a typical laptop. At 30 m resolution the output is usually tens of MB instead of hundreds of MB for a native 10 m full tile; remove `resolution=30` when you need native resolution.

In [ ]:
result = mosaic(
    grid_id="50HNG",
    start_year=2023,
    start_month=1,
    start_day=1,
    duration_days=14,
    output_dir=Path("output"),
    scene_order="valid_data",
    mosaic_method="first",
    bands=["visual"],
    cloud_mask="SCL",
    resolution=30,
    early_stop_missing_fraction=0.01,
    additional_query={"eo:cloud_cover": {"lt": 50}},
)
print(f"Visual mosaic saved to: {result}")

In [ ]:
with rio.open(result) as src:
    visual_array = src.read()

plot_rgb(visual_array, "50HNG visual mosaic at 30m")

metadata_path = result.with_suffix(".json")
metadata = json.loads(metadata_path.read_text())
summary = {
    "mode": metadata["mode"],
    "start_date": metadata["start_date"],
    "end_date": metadata["end_date"],
}
summary.update(
    {
        key: metadata["request"][key]
        for key in ["grid_id", "bands", "mosaic_method", "cloud_mask", "resolution"]
    }
)
print(summary)

## 2. Return raw bands as an array

If you do not pass `output_dir` or `output_path`, `mosaic()` returns an array and rasterio profile. This example includes visible, red-edge, NIR, and SWIR bands.

In [ ]:
bands = ["B04", "B03", "B02", "B06", "B08", "B11"]

array, rio_profile = mosaic(
    grid_id="50HMH",
    start_year=2022,
    start_month=1,
    start_day=1,
    duration_months=2,
    scene_order="valid_data",
    mosaic_method="mean",
    bands=bands,
    cloud_mask="SCL",
    resolution=30,
    early_stop_missing_fraction=0.01,
)

print(f"Mosaic array shape: {array.shape}")
print(f"CRS: {rio_profile['crs']}")
print(f"Pixel: {rio_profile['transform'].a}m")

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(10, 10))
for i, ax in enumerate(axs.flat):
    band = array[i]
    valid = band[np.isfinite(band) & (band > 0)]
    if valid.size:
        lo, hi = np.nanpercentile(valid, [2, 98])
        ax.imshow(band, vmin=lo, vmax=hi)
    else:
        ax.imshow(band)
    ax.set_title(f"Band {bands[i]}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()

## 3. Higher-quality percentile mosaic

This is intentionally heavier: it uses a longer time range and processes every selected image because percentile mosaics need multiple observations. Expect this to be the slowest cell in the notebook. Keep it for quality-sensitive work, or add `resolution=30` while experimenting.

In [ ]:
array, rio_profile = mosaic(
    grid_id="50HNG",
    start_year=2023,
    start_month=1,
    start_day=1,
    duration_months=3,
    scene_order="valid_data",
    mosaic_method="percentile",
    percentile=25,
    bands=["B04", "B03", "B02"],
    early_stop_missing_fraction=None,
    additional_query={"eo:cloud_cover": {"lt": 10}},
)

In [ ]:
plot_rgb(array, "50HNG percentile-25 mosaic")